In [ ]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# Import basic modules
import pandas as pd
from datum_api_client import DatumApi
import datetime
from datetime import timedelta
from typing import Optional, List, Dict, Any


# Import warnings
import warnings
warnings.filterwarnings("ignore")
# pip install xlrd
# pip install openpyxl

In [ ]:
from __future__ import annotations

# ── Pump&Dump: find violent afternoon moves and describe what preceded them ───────────────
#
# Unlike the other strategies here this one works on PRICE (column c), not on Stack%. A 10%
# move has to mean 10% from where the ticker actually was, and Stack% is a cumulative move
# against the previous close: a ticker sitting at +50% that goes to +65% has added 15 Stack%
# points but only moved 10% in price. Using Stack% differences would over-count every ticker
# that was already far from its previous close — which is exactly the pump&dump population.


def pumpdump_scan(
    input_path: str,
    *,
    output_events_jsonl: str = "PUMPDUMP/events.jsonl",
    output_daily_csv: str = "PUMPDUMP/daily.csv",
    # event definition: |price move| >= move_pct within any window_minutes, both ends inside
    # [scan_from, scan_to]
    move_pct: float = 10.0,
    window_minutes: int = 30,
    scan_from: tuple = (14, 0),
    scan_to: tuple = (16, 0),
    # forward horizons measured from the DETECTION bar (the moment the move completes)
    horizons_min: tuple = (15, 30, 60),
    close_hm: tuple = (16, 0),
    # a session runs 04:00 -> 03:59 next day, so the overnight block belongs to the session
    # that started the previous calendar morning (same convention as DayTwo)
    session_rollover_min: int = 240,
    # the reference time the "what did this ticker look like earlier today" snapshot is taken
    # at. Recorded for EVERY ticker-day, event or not, so pump days can be compared against a
    # control group on identical footing.
    snapshot_hm: tuple = (14, 0),
    # noise guards
    min_price: float = 0.0,          # e.g. 1.0 to drop sub-dollar tickers
    min_bars_before: int = 5,        # need some pre-history to describe
    min_day_volume: float = 0.0,
    write_daily: bool = True,
    daily_every_n: int = 1,          # keep every Nth ticker-day in daily.csv (memory lever)
    log_every_n_groups: int = 200,
):
    """
    Finds every situation where, after scan_from, a ticker moved more than move_pct in either
    direction inside a window_minutes window, and records:

      * the event itself (direction, size, start/end time, how far it eventually ran)
      * what the ticker looked like BEFORE it happened, both at a fixed 14:00 snapshot and at
        the instant the move started
      * what happened AFTER, at several horizons — so the same file answers both
        "what do these have in common" and "was there money in it"

    Detection is a sliding time window with a monotonic max/min deque: an event fires at the
    first bar whose price differs by >= move_pct from the extreme of the preceding
    window_minutes. Once one fires the window is reset past it, so a single 40-minute pump is
    one event, not fifteen overlapping ones.
    """
    import gc, json, time, gzip, math
    from collections import deque
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq

    DAY = 24 * 60
    thr = move_pct / 100.0
    W = int(window_minutes)

    def _sm(hm):
        t = hm[0] * 60 + hm[1]
        return t if t >= session_rollover_min else t + DAY

    scan_lo, scan_hi = _sm(scan_from), _sm(scan_to)
    close_sm = _sm(close_hm)
    snap_sm = _sm(snapshot_hm)
    if scan_hi <= scan_lo:
        raise ValueError("scan_to must be after scan_from")

    for p in (output_events_jsonl, output_daily_csv):
        Path(p).parent.mkdir(parents=True, exist_ok=True)

    def _open_gz(path, mode="wt"):
        if str(path).lower().endswith(".gz"):
            return gzip.open(path, mode, encoding="utf-8", newline="\n", compresslevel=6)
        return open(path, mode.replace("t", ""), encoding="utf-8", newline="\n")

    ev_f = _open_gz(output_events_jsonl, "wt")

    DAILY_COLS = ["ticker", "date", "bench", "label", "n_bars", "price_snap", "prev_close",
                  "gap_pct", "stack_snap", "bench_snap", "devsig_snap", "beta", "corr",
                  "ret_open_to_snap", "range_pct_pre", "vol_pre", "vol_last30_ratio",
                  "bars_pre", "max_move30_pre", "atr_pct_pre", "dollar_vol_pre"]
    daily_rows = []
    daily_written = 0
    if write_daily:
        pd.DataFrame(columns=DAILY_COLS).to_csv(output_daily_csv, index=False, mode="w")

    def _js(x):
        if x is None: return None
        if isinstance(x, (np.floating, float)):
            return None if (math.isnan(x) or math.isinf(x)) else round(float(x), 6)
        if isinstance(x, (np.integer, int)): return int(x)
        if isinstance(x, (np.bool_, bool)): return bool(x)
        return x

    def _detect(tm, px):
        """-> [(i_from, i_to, direction, move)] — one entry per distinct move."""
        out = []
        n = tm.size
        i = int(np.searchsorted(tm, scan_lo, "left"))
        dq_hi, dq_lo = deque(), deque()
        left = i
        while i < n and tm[i] <= scan_hi:
            while left < i and tm[left] < tm[i] - W:
                left += 1
            while dq_hi and dq_hi[0] < left: dq_hi.popleft()
            while dq_lo and dq_lo[0] < left: dq_lo.popleft()
            hit = None
            if dq_lo:
                j = dq_lo[0]
                up = px[i] / px[j] - 1.0
                if up >= thr:
                    hit = (j, i, 1, up)
            if hit is None and dq_hi:
                j = dq_hi[0]
                dn = px[i] / px[j] - 1.0
                if dn <= -thr:
                    hit = (j, i, -1, dn)
            if hit is not None:
                out.append(hit)
                # reset past the move so one long pump is one event, not a cascade
                dq_hi.clear(); dq_lo.clear()
                i += 1
                left = i
                continue
            while dq_hi and px[dq_hi[-1]] <= px[i]: dq_hi.pop()
            dq_hi.append(i)
            while dq_lo and px[dq_lo[-1]] >= px[i]: dq_lo.pop()
            dq_lo.append(i)
            i += 1
        return out

    def _at(tm, arr, target, side="last"):
        """value at the last bar <= target ('last') or first bar >= target ('next')."""
        if tm.size == 0: return None, None
        if side == "last":
            k = int(np.searchsorted(tm, target, "right")) - 1
            if k < 0: return None, None
        else:
            k = int(np.searchsorted(tm, target, "left"))
            if k >= tm.size: return None, None
        return float(arr[k]), k

    n_groups = n_events = 0
    t0 = time.time()

    def _process(tk, sdate, tm, px, hi, lo, vol, stack, bnch, dsig, bench_lbl, beta, corr,
                 prev_close, day_open):
        nonlocal n_events, daily_written
        n = tm.size
        if n < min_bars_before:
            return
        if min_day_volume > 0 and float(np.nansum(vol)) < min_day_volume:
            return

        events = _detect(tm, px)

        # ── snapshot state at 14:00 (same for event and control days) ──
        p_snap, k_snap = _at(tm, px, snap_sm)
        row = None
        if p_snap is not None and k_snap >= min_bars_before and (min_price <= 0 or p_snap >= min_price):
            pre = slice(0, k_snap + 1)
            v_pre = np.nan_to_num(vol[pre])
            tot_v = float(v_pre.sum())
            # volume in the last 30 min before the snapshot vs the day's average 30 min
            m30 = tm[pre] >= (snap_sm - 30)
            v30 = float(v_pre[m30].sum())
            span = max(float(tm[k_snap] - tm[0]), 1.0)
            v30_ratio = (v30 / max(tot_v * 30.0 / span, 1e-9)) if tot_v > 0 else None
            rng = (float(np.nanmax(hi[pre])) - float(np.nanmin(lo[pre]))) / prev_close * 100.0 \
                if prev_close and prev_close > 0 else None
            # biggest 30-minute swing already seen earlier in the day
            mm = 0.0
            pp = px[pre]; tt = tm[pre]
            j0 = 0
            for j1 in range(pp.size):
                while tt[j0] < tt[j1] - 30: j0 += 1
                seg = pp[j0:j1 + 1]
                if seg.size > 1:
                    mm = max(mm, float(seg.max() / seg.min() - 1.0))
            tr = np.abs(hi[pre] - lo[pre])
            row = {
                "ticker": tk, "date": sdate, "bench": bench_lbl,
                "label": "none", "n_bars": int(n), "price_snap": _js(p_snap),
                "prev_close": _js(prev_close),
                "gap_pct": _js((day_open / prev_close - 1.0) * 100.0 if (day_open and prev_close) else None),
                "stack_snap": _js(_at(tm, stack, snap_sm)[0]),
                "bench_snap": _js(_at(tm, bnch, snap_sm)[0]),
                "devsig_snap": _js(_at(tm, dsig, snap_sm)[0]),
                "beta": _js(beta), "corr": _js(corr),
                "ret_open_to_snap": _js((p_snap / day_open - 1.0) * 100.0 if day_open else None),
                "range_pct_pre": _js(rng),
                "vol_pre": _js(tot_v),
                "vol_last30_ratio": _js(v30_ratio),
                "bars_pre": int(k_snap + 1),
                "max_move30_pre": _js(mm * 100.0),
                "atr_pct_pre": _js(float(np.nanmean(tr)) / prev_close * 100.0 if prev_close else None),
                "dollar_vol_pre": _js(tot_v * float(np.nanmean(px[pre]))),
            }

        for (i0, i1, direction, move) in events:
            if min_price > 0 and px[i0] < min_price:
                continue
            if i0 < min_bars_before:
                continue
            pre = slice(0, i0 + 1)
            v_pre = np.nan_to_num(vol[pre])
            tot_v = float(v_pre.sum())
            m30 = tm[pre] >= (tm[i0] - 30)
            v30 = float(v_pre[m30].sum())
            span = max(float(tm[i0] - tm[0]), 1.0)
            e = {
                "ticker": tk, "date": sdate, "bench": bench_lbl,
                "dir": int(direction), "kind": "PUMP" if direction > 0 else "DUMP",
                "t_start": int(tm[i0] % DAY), "t_end": int(tm[i1] % DAY),
                "minutes": int(tm[i1] - tm[i0]),
                "move_pct": _js(move * 100.0),
                "price_start": _js(px[i0]), "price_end": _js(px[i1]),
                "prev_close": _js(prev_close),
                "stack_start": _js(float(stack[i0])) if np.isfinite(stack[i0]) else None,
                "bench_start": _js(float(bnch[i0])) if np.isfinite(bnch[i0]) else None,
                "devsig_start": _js(float(dsig[i0])) if np.isfinite(dsig[i0]) else None,
                "beta": _js(beta), "corr": _js(corr),
                # ── what preceded it ──
                "pre_bars": int(i0 + 1),
                "pre_ret_open": _js((px[i0] / day_open - 1.0) * 100.0 if day_open else None),
                "pre_gap_pct": _js((day_open / prev_close - 1.0) * 100.0 if (day_open and prev_close) else None),
                "pre_range_pct": _js((float(np.nanmax(hi[pre])) - float(np.nanmin(lo[pre]))) /
                                     prev_close * 100.0 if prev_close else None),
                "pre_vol": _js(tot_v),
                "pre_vol30_ratio": _js((v30 / max(tot_v * 30.0 / span, 1e-9)) if tot_v > 0 else None),
                "pre_dollar_vol": _js(tot_v * float(np.nanmean(px[pre]))),
                "pre_ret30": _js((px[i0] / _at(tm, px, tm[i0] - 30)[0] - 1.0) * 100.0
                                 if _at(tm, px, tm[i0] - 30)[0] else None),
                "pre_ret60": _js((px[i0] / _at(tm, px, tm[i0] - 60)[0] - 1.0) * 100.0
                                 if _at(tm, px, tm[i0] - 60)[0] else None),
            }
            # ── what happened after (from the DETECTION bar — that is when you could act) ──
            for hz in horizons_min:
                p, _k = _at(tm, px, tm[i1] + hz)
                e[f"fwd_{hz}m"] = _js((p / px[i1] - 1.0) * 100.0) if p else None
            p_cl, _k = _at(tm, px, close_sm)
            e["fwd_close"] = _js((p_cl / px[i1] - 1.0) * 100.0) if p_cl else None
            # how much further the move ran, and how much it gave back, after detection
            post = px[i1:]
            if post.size > 1:
                e["post_max_pct"] = _js((float(post.max()) / px[i1] - 1.0) * 100.0)
                e["post_min_pct"] = _js((float(post.min()) / px[i1] - 1.0) * 100.0)
            ev_f.write(json.dumps(e, ensure_ascii=False) + "\n")
            n_events += 1
            if row is not None:
                row["label"] = "PUMP" if direction > 0 else "DUMP"

        if row is not None and write_daily:
            daily_written += 1
            if daily_written % daily_every_n == 0:
                daily_rows.append(row)

    # ── streaming read, carrying the trailing partial (ticker, session) across row groups ──
    pf = pq.ParquetFile(input_path)
    need = ["ticker", "dt", "c", "h", "l", "v", "prev_close", "open", "Stack%",
            "bench", "corr", "beta", "Bench%", "dev_sig"]
    cols = [c for c in need if c in pf.schema.names]
    missing = {"ticker", "dt", "c"} - set(cols)
    if missing:
        raise KeyError(f"final.parquet is missing required columns: {sorted(missing)}")
    print(f"START Pump&Dump  file={input_path}")
    print(f"  move>={move_pct}% within {window_minutes}min, both ends in "
          f"[{scan_from[0]:02d}:{scan_from[1]:02d}, {scan_to[0]:02d}:{scan_to[1]:02d}]")
    print(f"  columns used: {cols}")

    carry = None
    try:
        for ci in range(pf.num_row_groups):
            df = pf.read_row_group(ci, columns=cols).to_pandas()
            if carry is not None and len(carry):
                df = pd.concat([carry, df], ignore_index=True)
                carry = None
            dt = pd.to_datetime(df["dt"], errors="coerce")
            ok = dt.notna().to_numpy(copy=False)
            df = df.loc[ok]; dt = dt[ok]
            if df.empty:
                continue
            t = (dt.dt.hour.to_numpy(dtype="int32") * 60 + dt.dt.minute.to_numpy(dtype="int32"))
            late = t < session_rollover_min
            smin = np.where(late, t + DAY, t).astype("int32")
            sess = dt - pd.Timedelta(minutes=session_rollover_min)
            sdate = sess.dt.strftime("%Y-%m-%d").to_numpy()
            tk = df["ticker"].to_numpy()

            # group boundaries on (ticker, session date)
            chg = np.empty(len(df), bool); chg[0] = True
            chg[1:] = (tk[1:] != tk[:-1]) | (sdate[1:] != sdate[:-1])
            starts = np.flatnonzero(chg)
            ends = np.append(starts[1:], len(df))

            def col(name, dflt=np.nan):
                return (pd.to_numeric(df[name], errors="coerce").to_numpy(dtype="float64")
                        if name in df.columns else np.full(len(df), dflt))
            c_ = col("c"); h_ = col("h"); l_ = col("l"); v_ = col("v")
            pc_ = col("prev_close"); op_ = col("open")
            st_ = col("Stack%"); bn_ = col("Bench%"); ds_ = col("dev_sig")
            be_ = col("beta"); co_ = col("corr")
            bl_ = df["bench"].astype(str).to_numpy() if "bench" in df.columns else np.array([""] * len(df))

            last = len(starts) - 1
            for gi, (s, e) in enumerate(zip(starts, ends)):
                if gi == last and ci + 1 < pf.num_row_groups:
                    carry = df.iloc[s:e].copy()      # may continue in the next row group
                    break
                o = np.argsort(smin[s:e], kind="stable")
                sl = slice(s, e)
                pxg = c_[sl][o]
                if not np.isfinite(pxg).all() or (pxg <= 0).any():
                    keep = np.isfinite(pxg) & (pxg > 0)
                    if keep.sum() < min_bars_before:
                        continue
                else:
                    keep = slice(None)
                _process(
                    str(tk[s]), str(sdate[s]),
                    smin[sl][o][keep].astype(np.int32), pxg[keep],
                    h_[sl][o][keep], l_[sl][o][keep], v_[sl][o][keep],
                    st_[sl][o][keep], bn_[sl][o][keep], ds_[sl][o][keep],
                    str(bl_[s]),
                    float(be_[s]) if np.isfinite(be_[s]) else None,
                    float(co_[s]) if np.isfinite(co_[s]) else None,
                    float(pc_[s]) if np.isfinite(pc_[s]) else None,
                    float(op_[s]) if np.isfinite(op_[s]) else None,
                )
                n_groups += 1

            if daily_rows and len(daily_rows) >= 20000:
                pd.DataFrame(daily_rows, columns=DAILY_COLS).to_csv(
                    output_daily_csv, mode="a", header=False, index=False)
                daily_rows.clear()
            if (ci + 1) % log_every_n_groups == 0:
                el = time.time() - t0
                print(f"[rg {ci+1:>5}/{pf.num_row_groups}] ticker-days={n_groups:,} "
                      f"events={n_events:,} elapsed={el:.0f}s")
                gc.collect()
        if carry is not None and len(carry):
            # flush the tail through one more (single row group) pass
            dt = pd.to_datetime(carry["dt"], errors="coerce")
            carry = carry.loc[dt.notna()]
            if len(carry):
                dt = pd.to_datetime(carry["dt"])
                t = dt.dt.hour.to_numpy(dtype="int32") * 60 + dt.dt.minute.to_numpy(dtype="int32")
                smin = np.where(t < session_rollover_min, t + DAY, t).astype("int32")
                o = np.argsort(smin, kind="stable")
                g = lambda nm: (pd.to_numeric(carry[nm], errors="coerce").to_numpy(dtype="float64")[o]
                                if nm in carry.columns else np.full(len(carry), np.nan))
                px = g("c")
                keep = np.isfinite(px) & (px > 0)
                if keep.sum() >= min_bars_before:
                    _process(str(carry["ticker"].iloc[0]),
                             (dt - pd.Timedelta(minutes=session_rollover_min)).dt.strftime("%Y-%m-%d").iloc[0],
                             smin[o][keep], px[keep], g("h")[keep], g("l")[keep], g("v")[keep],
                             g("Stack%")[keep], g("Bench%")[keep], g("dev_sig")[keep],
                             str(carry["bench"].iloc[0]) if "bench" in carry.columns else "",
                             float(g("beta")[0]) if np.isfinite(g("beta")[0]) else None,
                             float(g("corr")[0]) if np.isfinite(g("corr")[0]) else None,
                             float(g("prev_close")[0]) if np.isfinite(g("prev_close")[0]) else None,
                             float(g("open")[0]) if np.isfinite(g("open")[0]) else None)
                    n_groups += 1
    finally:
        if daily_rows:
            pd.DataFrame(daily_rows, columns=DAILY_COLS).to_csv(
                output_daily_csv, mode="a", header=False, index=False)
        ev_f.close()

    print(f"DONE ticker-days={n_groups:,} events={n_events:,} elapsed={time.time()-t0:.0f}s")
    print(f"  events = {output_events_jsonl}")
    print(f"  daily  = {output_daily_csv}")
    return {"ticker_days": n_groups, "events": n_events}

In [ ]:
# ── Pump&Dump: what do they have in common, and was there money in it ─────────────────────


def pumpdump_report(
    events_jsonl: str,
    daily_csv: str = None,
    *,
    horizons_min: tuple = (15, 30, 60),
    top_features: int = 12,
    min_group: int = 20,
):
    """
    Two questions, answered separately for PUMPs and DUMPs.

    1. COMMON TRAITS — every pre-event feature is scored by AUC against the control group
       (ticker-days that never produced an event). AUC 0.5 = the feature says nothing;
       0.8 = it separates well. Rank-based, so outliers and non-normal distributions are
       harmless. Requires daily_csv.

    2. PROFIT — the forward return from the DETECTION bar (the first moment you could have
       acted), for both directions of trade:
         FOLLOW = keep going with the move,  FADE = bet on the snap-back.
       Reported as the SUM of per-event returns in percent — i.e. what a fixed-size position
       in every single event would have added up to, before costs.
    """
    import json, gzip, math
    import numpy as np
    import pandas as pd

    def _read(p):
        op = gzip.open if str(p).lower().endswith(".gz") else open
        with op(p, "rt", encoding="utf-8") as f:
            return pd.DataFrame([json.loads(l) for l in f if l.strip()])

    ev = _read(events_jsonl)
    if ev.empty:
        print("no events found")
        return None

    print("=" * 78)
    print(f"EVENTS: {len(ev):,}   PUMP={int((ev['dir'] > 0).sum()):,}   "
          f"DUMP={int((ev['dir'] < 0).sum()):,}   "
          f"tickers={ev.ticker.nunique():,}   days={ev.date.nunique()}")
    print(f"move size %: p50={ev.move_pct.abs().median():.1f}  p90={ev.move_pct.abs().quantile(.9):.1f}  "
          f"max={ev.move_pct.abs().max():.1f}")
    print(f"minutes to complete: p50={ev.minutes.median():.0f}  p90={ev.minutes.quantile(.9):.0f}")
    hh = (ev.t_start // 60).astype(int)
    print("start hour:", hh.value_counts().sort_index().to_dict())

    # ── 1. common traits ────────────────────────────────────────────────────
    if daily_csv:
        dl = pd.read_csv(daily_csv)
        ctrl = dl[dl.label == "none"]
        feats = ["price_snap", "gap_pct", "stack_snap", "bench_snap", "devsig_snap",
                 "beta", "corr", "ret_open_to_snap", "range_pct_pre", "vol_pre",
                 "vol_last30_ratio", "bars_pre", "max_move30_pre", "atr_pct_pre",
                 "dollar_vol_pre"]
        feats = [f for f in feats if f in dl.columns]

        def auc(pos, neg):
            pos = pos[np.isfinite(pos)]; neg = neg[np.isfinite(neg)]
            if pos.size < min_group or neg.size < min_group: return None, None, None
            allv = np.concatenate([pos, neg])
            r = pd.Series(allv).rank().to_numpy()
            a = (r[:pos.size].sum() - pos.size * (pos.size + 1) / 2) / (pos.size * neg.size)
            return a, float(np.median(pos)), float(np.median(neg))

        for lab in ("PUMP", "DUMP"):
            grp = dl[dl.label == lab]
            if len(grp) < min_group:
                print(f"\n{lab}: only {len(grp)} labelled ticker-days — too few to profile")
                continue
            rows = []
            for f in feats:
                a, mp, mn = auc(grp[f].to_numpy(dtype=float), ctrl[f].to_numpy(dtype=float))
                if a is None: continue
                rows.append({"feature": f, "AUC": round(a, 3), "|AUC-.5|": round(abs(a - .5), 3),
                             f"median_{lab}": round(mp, 4), "median_control": round(mn, 4)})
            t = pd.DataFrame(rows).sort_values("|AUC-.5|", ascending=False).head(top_features)
            print(f"\n--- what a {lab} day looks like at the 14:00 snapshot "
                  f"(n={len(grp):,} vs control n={len(ctrl):,}) ---")
            print(t.drop(columns="|AUC-.5|").to_string(index=False))
    else:
        print("\n(no daily_csv given — skipping the common-traits comparison)")

    # ── 2. profit ───────────────────────────────────────────────────────────
    cols = [f"fwd_{h}m" for h in horizons_min] + ["fwd_close"]
    cols = [c for c in cols if c in ev.columns]
    print("\n" + "=" * 78)
    print("FORWARD RETURN FROM THE DETECTION BAR  (sum of per-event %, equal size, no costs)")
    print("  FOLLOW = trade with the move (long a PUMP / short a DUMP)")
    print("  FADE   = trade against it")
    out = []
    for lab, sub in (("PUMP", ev[ev.dir > 0]), ("DUMP", ev[ev.dir < 0]), ("ALL", ev)):
        for c in cols:
            r = pd.to_numeric(sub[c], errors="coerce").dropna()
            if r.empty: continue
            # FOLLOW: a dump is traded short, so its P&L is the negated price return
            sgn = sub.loc[r.index, "dir"].to_numpy()
            follow = r.to_numpy() * sgn
            out.append({"group": lab, "horizon": c, "n": len(r),
                        "FOLLOW_total%": round(follow.sum(), 1),
                        "FOLLOW_mean%": round(follow.mean(), 3),
                        "FOLLOW_win%": round((follow > 0).mean() * 100, 1),
                        "FADE_total%": round(-follow.sum(), 1),
                        "FADE_mean%": round(-follow.mean(), 3),
                        "FADE_win%": round((follow < 0).mean() * 100, 1)})
    prof = pd.DataFrame(out)
    print(prof.to_string(index=False))

    best = prof.loc[prof[["FOLLOW_total%", "FADE_total%"]].max(axis=1).idxmax()]
    side = "FOLLOW" if best["FOLLOW_total%"] >= best["FADE_total%"] else "FADE"
    print(f"\nBEST: {side} on {best['group']} at {best['horizon']} -> "
          f"{best[side + '_total%']:.1f}% total over {int(best['n'])} events "
          f"({best[side + '_mean%']:.3f}% per event, {best[side + '_win%']:.0f}% win rate)")
    print("Reminder: no slippage, no borrow cost, and a 10% mover is exactly the kind of name")
    print("where both are largest — treat these totals as an upper bound.")
    return {"events": ev, "profit": prof}

In [ ]:
# ── Pump&Dump v2: overnight behaviour -> what happens after 07:00 ─────────────────────────
#
# Everything here is strictly separated in time, which the first version was not:
#   OBSERVE  [prev session 15:00 .. 07:00]   — how the Stack% behaved overnight
#   EVALUATE (07:00 .. 16:00]                — the move that follows
# No feature can see past 07:00 and no outcome starts before it, so any pattern found is a
# genuine forecast rather than a description of a move that had already happened.
#
# "Previous day" means the previous session PRESENT IN THE DATA for that ticker, not the
# previous calendar date — otherwise every Monday would look at an empty Sunday. The real
# span is recorded as obs_hours so weekends can be separated out afterwards.


def pumpdump_overnight_scan(
    input_path: str,
    *,
    output_csv: str = "PUMPDUMP/overnight.csv",
    obs_from_hm: tuple = (15, 0),      # on the PREVIOUS session
    ref_hm: tuple = (7, 0),            # observation ends / evaluation starts
    fwd_to_hm: tuple = (16, 0),
    move_pct: float = 30.0,            # what counts as a violent move after ref_hm
    # the 07:00 reference must be a reasonably fresh print, otherwise the "move after 07:00"
    # silently swallows an overnight gap that happened hours earlier
    ref_max_stale_min: int = 180,
    horizons_min: tuple = (60, 150),   # 08:00 and 09:30 from a 07:00 reference
    # CONFIRMATION ENTRY. Buying every screened name at 07:00 pays the spread on all of them
    # while only a few percent ever move; instead wait for the move to actually start.
    # trigger_pct = how far from the 07:00 print the move must go before entering (entry is
    # taken at the CLOSE of the bar that crossed it, so any overshoot is charged as slippage).
    # stop_pct = give-back from the entry that closes the trade. target_pct = None -> ride to
    # the end of the window. Both sides are simulated independently on every ticker-day.
    trigger_pct: float = 8.0,
    stop_pct: float = 8.0,
    target_pct: float = None,
    min_obs_bars: int = 3,
    min_fwd_bars: int = 5,
    log_every_n_groups: int = 500,
):
    import gc, time
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import pyarrow.parquet as pq

    DAY = 1440
    OBS0 = obs_from_hm[0] * 60 + obs_from_hm[1]
    REF = ref_hm[0] * 60 + ref_hm[1]
    FWD1 = fwd_to_hm[0] * 60 + fwd_to_hm[1]
    thr = move_pct / 100.0

    # The overnight block split into its two halves, because the shape matters and a single
    # 15:00->07:00 aggregate hides it: a ticker that ran up in the evening and then bled all
    # night looks identical to a flat one once you only measure the endpoints.
    LEG_COLS = ["st_1600", "st_0000", "st_0700", "leg_pm_pp", "leg_on_pp",
                "leg_pm_ret", "leg_on_ret", "leg_pm_bars", "leg_on_bars"]
    COLS = ["ticker", "date", "bench", "beta", "corr", "prev_close", "price_ref",
            "ref_stale_min", "obs_hours", "obs_bars", "obs_gap_days",
            "on_stack_first", "on_stack_ref", "on_stack_min", "on_stack_max",
            "on_drift", "on_range", "on_up", "on_dn", "on_std", "on_up_frac",
            "on_t_max_frac", "on_t_min_frac", "on_last_leg", "on_ret", "on_vol",
            "on_dollar_vol", "on_night_only_bars",
            "fwd_bars", "fwd_max_pct", "fwd_min_pct", "fwd_close_pct",
            "event", "event_dir", "t_hit"] + LEG_COLS + [f"fwd_{h}m_pct" for h in horizons_min] + [
            f"{p}_{s}" for p in ("trg_up", "trg_dn")
            for s in ("hit", "t", "px", "exit_pct", "mfe", "mae", "stop_pct")]
    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(columns=COLS).to_csv(output_csv, index=False, mode="w")

    buf = []
    n_rows_out = 0
    n_units = 0
    n_events = 0
    t0 = time.time()

    def _f(x):
        try:
            v = float(x)
            return None if (np.isnan(v) or np.isinf(v)) else round(v, 6)
        except Exception:
            return None

    def _flush_ticker(tk, amin, px, vol, stack, bench_lbl, beta, corr, pclose):
        """amin = absolute minute (day_ordinal*1440 + minute of day), already sorted."""
        nonlocal n_units, n_events
        if amin.size < min_obs_bars + min_fwd_bars:
            return
        day = amin // DAY
        udays = np.unique(day)
        if udays.size < 2:
            return
        rows = []
        for k in range(1, udays.size):
            D = int(udays[k])
            Dp = int(udays[k - 1])                      # previous session present in the data
            o_lo = Dp * DAY + OBS0
            o_hi = D * DAY + REF
            i0 = int(np.searchsorted(amin, o_lo, "left"))
            i1 = int(np.searchsorted(amin, o_hi, "right"))     # exclusive
            j1 = int(np.searchsorted(amin, D * DAY + FWD1, "right"))
            if i1 - i0 < min_obs_bars or j1 - i1 < min_fwd_bars:
                continue
            ref_i = i1 - 1
            stale = int(o_hi - amin[ref_i])
            if stale > ref_max_stale_min:
                continue
            p0 = px[ref_i]
            if not np.isfinite(p0) or p0 <= 0:
                continue

            ob = slice(i0, i1)
            st = stack[ob]
            st_ok = st[np.isfinite(st)]
            if st_ok.size < min_obs_bars:
                continue
            t_rel = (amin[ob] - o_lo).astype(float)
            span = max(float(t_rel[-1]), 1.0)
            d_st = np.diff(st_ok)
            night_only = int(np.sum((amin[ob] % DAY >= 20 * 60) | (amin[ob] % DAY < 4 * 60)))
            leg_lo = D * DAY + 4 * 60
            leg_i = int(np.searchsorted(amin, leg_lo, "left"))
            last_leg = (float(stack[ref_i] - stack[leg_i])
                        if leg_i < i1 and np.isfinite(stack[leg_i]) and np.isfinite(stack[ref_i])
                        else None)

            # ── evening leg (16:00 -> 00:00) and night leg (00:00 -> 07:00) ──
            def _last_at(target):
                kk = int(np.searchsorted(amin, target, "right")) - 1
                return kk if (i0 <= kk < i1) else None
            k16 = _last_at(Dp * DAY + 16 * 60)
            k00 = _last_at(D * DAY)
            k07 = ref_i
            def _sv(kk):
                return float(stack[kk]) if (kk is not None and np.isfinite(stack[kk])) else None
            s16, s00, s07 = _sv(k16), _sv(k00), _sv(k07)
            leg_pm_pp = (s00 - s16) if (s16 is not None and s00 is not None) else None
            leg_on_pp = (s07 - s00) if (s00 is not None and s07 is not None) else None
            leg_pm_ret = ((px[k00] / px[k16] - 1.0) * 100.0
                          if (k16 is not None and k00 is not None and px[k16] > 0) else None)
            leg_on_ret = ((px[k07] / px[k00] - 1.0) * 100.0
                          if (k00 is not None and px[k00] > 0) else None)
            leg_pm_bars = (int(k00 - k16) if (k16 is not None and k00 is not None) else 0)
            leg_on_bars = (int(k07 - k00) if k00 is not None else 0)

            fw = slice(i1, j1)
            fp = px[fw]
            fp = fp[np.isfinite(fp) & (fp > 0)]
            if fp.size < min_fwd_bars:
                continue
            up = float(fp.max() / p0 - 1.0)
            dn = float(fp.min() / p0 - 1.0)
            ev = 1 if max(up, -dn) >= thr else 0
            edir = 0
            t_hit = None
            if ev:
                edir = 1 if up >= -dn else -1
                r = px[fw] / p0 - 1.0
                hit = np.flatnonzero(r >= thr) if edir > 0 else np.flatnonzero(r <= -thr)
                if hit.size:
                    t_hit = int(amin[i1 + int(hit[0])] % DAY)
                n_events += 1

            row = {
                "ticker": tk, "date": f"{pd.Timestamp.fromordinal(D):%Y-%m-%d}",
                "bench": bench_lbl, "beta": _f(beta), "corr": _f(corr),
                "prev_close": _f(pclose), "price_ref": _f(p0), "ref_stale_min": stale,
                "obs_hours": round((amin[i1 - 1] - amin[i0]) / 60.0, 2),
                "obs_bars": int(i1 - i0), "obs_gap_days": int(D - Dp),
                "on_stack_first": _f(st_ok[0]), "on_stack_ref": _f(st_ok[-1]),
                "on_stack_min": _f(st_ok.min()), "on_stack_max": _f(st_ok.max()),
                "on_drift": _f(st_ok[-1] - st_ok[0]),
                "on_range": _f(st_ok.max() - st_ok.min()),
                "on_up": _f(st_ok.max() - st_ok[0]), "on_dn": _f(st_ok[0] - st_ok.min()),
                "on_std": _f(d_st.std()) if d_st.size > 1 else None,
                "on_up_frac": _f((d_st > 0).mean()) if d_st.size else None,
                "on_t_max_frac": _f(t_rel[int(np.nanargmax(np.where(np.isfinite(st), st, -np.inf)))] / span),
                "on_t_min_frac": _f(t_rel[int(np.nanargmin(np.where(np.isfinite(st), st, np.inf)))] / span),
                "on_last_leg": _f(last_leg),
                "on_ret": _f((p0 / px[i0] - 1.0) * 100.0) if px[i0] > 0 else None,
                "on_vol": _f(np.nansum(vol[ob])),
                "on_dollar_vol": _f(np.nansum(vol[ob] * px[ob])),
                "on_night_only_bars": night_only,
                "fwd_bars": int(fp.size),
                "fwd_max_pct": _f(up * 100.0), "fwd_min_pct": _f(dn * 100.0),
                "fwd_close_pct": _f((px[j1 - 1] / p0 - 1.0) * 100.0),
                "event": ev, "event_dir": edir, "t_hit": t_hit,
                "st_1600": _f(s16), "st_0000": _f(s00), "st_0700": _f(s07),
                "leg_pm_pp": _f(leg_pm_pp), "leg_on_pp": _f(leg_on_pp),
                "leg_pm_ret": _f(leg_pm_ret), "leg_on_ret": _f(leg_on_ret),
                "leg_pm_bars": leg_pm_bars, "leg_on_bars": leg_on_bars,
            }
            for h in horizons_min:
                kk = int(np.searchsorted(amin, D * DAY + REF + h, "right")) - 1
                row[f"fwd_{h}m_pct"] = _f((px[kk] / p0 - 1.0) * 100.0) if kk >= i1 else None

            # ── confirmation entry, simulated bar by bar on both sides ──
            fpx = px[i1:j1]
            ftm = amin[i1:j1]
            for sgn, pref in ((1, "trg_up"), (-1, "trg_dn")):
                lvl = p0 * (1.0 + sgn * trigger_pct / 100.0)
                hit = np.flatnonzero(fpx >= lvl) if sgn > 0 else np.flatnonzero(fpx <= lvl)
                if hit.size == 0:
                    row[f"{pref}_hit"] = 0
                    for s in ("t", "px", "exit_pct", "mfe", "mae", "stop_pct"):
                        row[f"{pref}_{s}"] = None
                    continue
                k = int(hit[0])
                ep = float(fpx[k])
                r = sgn * (fpx[k:] / ep - 1.0) * 100.0      # P&L of the position, in %
                i_stop = np.flatnonzero(r <= -stop_pct)
                i_tgt = (np.flatnonzero(r >= target_pct) if target_pct is not None
                         else np.empty(0, dtype=np.int64))
                cand = [a[0] for a in (i_stop, i_tgt) if a.size]
                if cand:
                    f = int(min(cand))
                    closed = -stop_pct if (i_stop.size and f == i_stop[0]) else target_pct
                else:
                    closed = float(r[-1])
                row[f"{pref}_hit"] = 1
                row[f"{pref}_t"] = int(ftm[k] % DAY)
                row[f"{pref}_px"] = _f(ep)
                row[f"{pref}_exit_pct"] = _f(float(r[-1]))
                row[f"{pref}_mfe"] = _f(float(r.max()))
                row[f"{pref}_mae"] = _f(float(r.min()))
                row[f"{pref}_stop_pct"] = _f(float(closed))
            rows.append(row)
            n_units += 1
        if rows:
            buf.extend(rows)

    pf = pq.ParquetFile(input_path)
    need = ["ticker", "dt", "c", "v", "prev_close", "Stack%", "bench", "corr", "beta"]
    cols = [c for c in need if c in pf.schema.names]
    print(f"START Pump&Dump overnight  file={input_path}")
    print(f"  confirmation entry: trigger={trigger_pct}% stop={stop_pct}% target={target_pct}")
    print(f"  observe [prev {obs_from_hm[0]:02d}:{obs_from_hm[1]:02d} .. "
          f"{ref_hm[0]:02d}:{ref_hm[1]:02d}]  ->  evaluate move >= {move_pct}% until "
          f"{fwd_to_hm[0]:02d}:{fwd_to_hm[1]:02d}")

    cur_tk = None
    parts = []

    def _emit():
        nonlocal parts, cur_tk, n_rows_out
        if cur_tk is None or not parts:
            return
        d = pd.concat(parts, ignore_index=True) if len(parts) > 1 else parts[0]
        dt = pd.to_datetime(d["dt"], errors="coerce")
        d = d.loc[dt.notna()]
        if len(d) >= min_obs_bars + min_fwd_bars:
            dt = pd.to_datetime(d["dt"])
            am = (dt.dt.normalize().map(lambda x: x.toordinal()).to_numpy(dtype=np.int64) * DAY
                  + dt.dt.hour.to_numpy(dtype=np.int64) * 60 + dt.dt.minute.to_numpy(dtype=np.int64))
            o = np.argsort(am, kind="stable")
            g = lambda nm: (pd.to_numeric(d[nm], errors="coerce").to_numpy(dtype="float64")[o]
                            if nm in d.columns else np.full(len(d), np.nan))
            pxv = g("c")
            keep = np.isfinite(pxv) & (pxv > 0)
            if keep.sum() >= min_obs_bars + min_fwd_bars:
                bl = str(d["bench"].iloc[0]) if "bench" in d.columns else ""
                bt = g("beta"); cr = g("corr"); pc = g("prev_close")
                _flush_ticker(cur_tk, am[o][keep], pxv[keep], g("v")[keep], g("Stack%")[keep],
                              bl,
                              float(bt[0]) if np.isfinite(bt[0]) else None,
                              float(cr[0]) if np.isfinite(cr[0]) else None,
                              float(pc[0]) if np.isfinite(pc[0]) else None)
        parts = []

    for ci in range(pf.num_row_groups):
        df = pf.read_row_group(ci, columns=cols).to_pandas()
        if df.empty:
            continue
        tk = df["ticker"].to_numpy()
        chg = np.flatnonzero(np.append(True, tk[1:] != tk[:-1]))
        ends = np.append(chg[1:], len(df))
        for s, e in zip(chg, ends):
            name = str(tk[s])
            if cur_tk is not None and name != cur_tk:
                _emit()
            cur_tk = name
            parts.append(df.iloc[s:e])
        if buf and len(buf) >= 20000:
            pd.DataFrame(buf, columns=COLS).to_csv(output_csv, mode="a", header=False, index=False)
            n_rows_out += len(buf); buf.clear()
        if (ci + 1) % log_every_n_groups == 0:
            print(f"[rg {ci+1:>5}/{pf.num_row_groups}] units={n_units:,} events={n_events:,} "
                  f"elapsed={time.time()-t0:.0f}s")
            gc.collect()
    _emit()
    if buf:
        pd.DataFrame(buf, columns=COLS).to_csv(output_csv, mode="a", header=False, index=False)
        n_rows_out += len(buf); buf.clear()

    print(f"DONE units={n_units:,} events={n_events:,} ({n_events/max(n_units,1)*100:.2f}%) "
          f"elapsed={time.time()-t0:.0f}s")
    print(f"  overnight table = {output_csv}")
    return {"units": n_units, "events": n_events}

In [ ]:
def pumpdump_overnight_report(
    overnight_csv: str,
    *,
    horizons=("fwd_60m_pct", "fwd_150m_pct", "fwd_close_pct"),
    min_price: float = 0.0,
    min_dollar_vol: float = 0.0,
    max_obs_gap_days: int = 3,
    top_features: int = 14,
):
    """AUC of every overnight feature against (a) any >=move_pct move, (b) up vs down; then
    the P&L of acting at 07:00 on the strongest rules."""
    import numpy as np
    import pandas as pd

    d = pd.read_csv(overnight_csv)
    d = d[d.obs_gap_days <= max_obs_gap_days]
    if min_price > 0: d = d[d.price_ref >= min_price]
    if min_dollar_vol > 0: d = d[d.on_dollar_vol >= min_dollar_vol]
    print("=" * 78)
    print(f"units={len(d):,}  tickers={d.ticker.nunique():,}  days={d.date.nunique()}")
    ev = d[d.event == 1]
    up = d[d.event_dir == 1]; dn = d[d.event_dir == -1]
    print(f"events={len(ev):,} ({len(ev)/max(len(d),1)*100:.2f}%)   UP={len(up):,}  DOWN={len(dn):,}")
    if len(ev):
        # t_hit is a CLOCK time (minutes since midnight), not an elapsed duration
        _q = np.nanpercentile(ev.t_hit, [25, 50, 90])
        print("threshold first crossed at (clock): p25={} p50={} p90={}".format(
            *[f"{int(v)//60:02d}:{int(v)%60:02d}" for v in _q]))

    FEATS = ["on_drift", "on_range", "on_up", "on_dn", "on_std", "on_up_frac", "on_ret",
             "on_stack_first", "on_stack_ref", "on_stack_min", "on_stack_max",
             "on_t_max_frac", "on_t_min_frac", "on_last_leg", "on_vol", "on_dollar_vol",
             "obs_bars", "on_night_only_bars", "price_ref", "beta", "corr", "ref_stale_min"]
    FEATS = [f for f in FEATS if f in d.columns]

    def auc(pos, neg):
        pos = pos[np.isfinite(pos)]; neg = neg[np.isfinite(neg)]
        if pos.size < 20 or neg.size < 20: return None, None, None
        r = pd.Series(np.concatenate([pos, neg])).rank().to_numpy()
        return ((r[:pos.size].sum() - pos.size * (pos.size + 1) / 2) / (pos.size * neg.size),
                float(np.median(pos)), float(np.median(neg)))

    for lab, pos, neg, title in (
        ("ANY", ev, d[d.event == 0], "any >=threshold move after 07:00"),
        ("UP", up, d[d.event == 0], "an UP move"),
        ("DOWN", dn, d[d.event == 0], "a DOWN move"),
        ("DIR", dn, up, "DOWN rather than UP (given a move happens)"),
    ):
        if len(pos) < 20 or len(neg) < 20:
            print(f"\n--- {title}: too few ({len(pos)} vs {len(neg)}) ---")
            continue
        rows = []
        for f in FEATS:
            a, mp, mn = auc(pos[f].to_numpy(float), neg[f].to_numpy(float))
            if a is None: continue
            rows.append({"feature": f, "AUC": round(a, 3), "|d|": round(abs(a - .5), 3),
                         "median_pos": round(mp, 3), "median_neg": round(mn, 3)})
        t = pd.DataFrame(rows).sort_values("|d|", ascending=False).head(top_features)
        print(f"\n--- overnight signature of {title}  (n={len(pos):,} vs {len(neg):,}) ---")
        print(t.drop(columns="|d|").to_string(index=False))

    print("\n" + "=" * 78)
    print("P&L OF ACTING AT 07:00 (enter at the 07:00 print, sum of per-trade %, no costs)")
    base = {h: d[h].dropna() for h in horizons if h in d.columns}
    for h, r in base.items():
        print(f"  buy-everything baseline {h}: total={r.sum():+,.0f}%  mean={r.mean():+.3f}%  n={len(r):,}")
    rules = {
        "on_drift >= +10 (ran up overnight)": d.on_drift >= 10,
        "on_drift <= -10 (sold off overnight)": d.on_drift <= -10,
        "on_range >= 20 (violent night)": d.on_range >= 20,
        "on_range >= 20 & drift>0": (d.on_range >= 20) & (d.on_drift > 0),
        "on_range >= 20 & drift<0": (d.on_range >= 20) & (d.on_drift < 0),
    }
    out = []
    for name, m in rules.items():
        sub = d[m.fillna(False)]
        if len(sub) < 20: continue
        for h in base:
            r = sub[h].dropna()
            if r.empty: continue
            out.append({"rule": name, "horizon": h, "n": len(r),
                        "LONG_total%": round(r.sum(), 0), "LONG_mean%": round(r.mean(), 3),
                        "SHORT_total%": round(-r.sum(), 0), "win_long%": round((r > 0).mean() * 100, 1),
                        "event_rate%": round(sub.event.mean() * 100, 2)})
    if out:
        print()
        print(pd.DataFrame(out).to_string(index=False))
    return d

In [ ]:
from pathlib import Path
import os


def _resolve_orion_paths(strategy_code: str):
    final_env = os.environ.get("FINAL_PARQUET_PATH")
    sig_env   = os.environ.get("SIGNALS_DIR")
    orion_env = os.environ.get("ORION_HOME")

    final_path   = Path(final_env).expanduser().resolve() if final_env else None
    signals_base = Path(sig_env).expanduser().resolve()   if sig_env   else None

    if (final_path is None or signals_base is None) and orion_env:
        orion_home = Path(orion_env).expanduser().resolve()
        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    if final_path is None or signals_base is None:
        here = Path.cwd().resolve()
        orion_home = None
        for parent in [here] + list(here.parents):
            if parent.name.lower() == "orion":
                orion_home = parent
                break
            cand = parent / "OriON"
            if cand.exists() and cand.is_dir():
                orion_home = cand.resolve()
                break

        if orion_home is None:
            raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var (recommended).")

        if final_path is None:
            final_path = (orion_home / "CRACEN" / "final.parquet").resolve()
        if signals_base is None:
            signals_base = (orion_home / "signals").resolve()

    out_dir = (signals_base / strategy_code.lower()).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    if not final_path.exists():
        raise FileNotFoundError(f"FINAL parquet not found: {final_path}")

    return final_path, out_dir


# ── runner ────────────────────────────────────────────────────────────────────

FINAL_PATH, OUT_DIR = _resolve_orion_paths("pumpdump")

pumpdump_scan(
    input_path=str(FINAL_PATH),
    output_events_jsonl=str(OUT_DIR / "events.jsonl.gz"),
    output_daily_csv=str(OUT_DIR / "daily.csv"),
    move_pct=10.0,
    window_minutes=30,
    scan_from=(14, 0), scan_to=(16, 0),
    horizons_min=(15, 30, 60),
    close_hm=(16, 0),
    snapshot_hm=(14, 0),
    min_price=0.0,              # e.g. 1.0 to exclude sub-dollar names
    min_bars_before=5,
    write_daily=True,
)

pumpdump_report(
    events_jsonl=str(OUT_DIR / "events.jsonl.gz"),
    daily_csv=str(OUT_DIR / "daily.csv"),
    horizons_min=(15, 30, 60),
)
